# Weather Condition Prediction Model
**Tea Plantation Smart Monitoring System**

This notebook trains a Gradient Boosting Classifier (GBC) to predict daily weather conditions from meteorological features.
Features: `precipitation`, `temp_max`, `temp_min`, `humidity_day`, `humidity_night`, `wind`
Target: `weather_label` (Sunny, Cloudy, Rainy, Stormy)

Artifacts saved: `weather_gbc.pkl`, `weather_scaler.pkl`, `weather_label_encoder.pkl`


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

ARTIFACTS_DIR = os.path.join(os.path.dirname(os.getcwd()), 'ml_training', 'model_artifacts')
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
print(f'Artifacts will be saved to: {ARTIFACTS_DIR}')


## 1. Data Generation
Synthetic meteorological dataset. Replace the block marked `# REPLACE WITH REAL DATA` with a `pd.read_csv()` call pointing to your actual dataset.


In [ ]:
# REPLACE WITH REAL DATA
# To use real data: df = pd.read_csv('path/to/weather_data.csv')
# Expected columns: precipitation, temp_max, temp_min, humidity_day, humidity_night, wind, weather_label

np.random.seed(42)
N = 2000

weather_labels = ['Sunny', 'Cloudy', 'Rainy', 'Stormy']
label_probs = [0.35, 0.30, 0.25, 0.10]
labels = np.random.choice(weather_labels, size=N, p=label_probs)

records = []
for lbl in labels:
    if lbl == 'Sunny':
        rec = dict(
            precipitation=np.random.uniform(0, 2),
            temp_max=np.random.normal(30, 3),
            temp_min=np.random.normal(20, 2),
            humidity_day=np.random.normal(45, 8),
            humidity_night=np.random.normal(55, 8),
            wind=np.random.normal(8, 3)
        )
    elif lbl == 'Cloudy':
        rec = dict(
            precipitation=np.random.uniform(0, 5),
            temp_max=np.random.normal(25, 3),
            temp_min=np.random.normal(18, 2),
            humidity_day=np.random.normal(65, 8),
            humidity_night=np.random.normal(72, 8),
            wind=np.random.normal(12, 4)
        )
    elif lbl == 'Rainy':
        rec = dict(
            precipitation=np.random.uniform(5, 30),
            temp_max=np.random.normal(22, 3),
            temp_min=np.random.normal(16, 2),
            humidity_day=np.random.normal(80, 6),
            humidity_night=np.random.normal(88, 5),
            wind=np.random.normal(18, 5)
        )
    else:  # Stormy
        rec = dict(
            precipitation=np.random.uniform(25, 80),
            temp_max=np.random.normal(19, 4),
            temp_min=np.random.normal(14, 3),
            humidity_day=np.random.normal(90, 5),
            humidity_night=np.random.normal(95, 3),
            wind=np.random.normal(35, 8)
        )
    rec['weather_label'] = lbl
    records.append(rec)

df = pd.DataFrame(records)

# Inject ~3% missing values to test NOCB fill
for col in ['precipitation', 'humidity_day', 'humidity_night']:
    mask = np.random.rand(N) < 0.03
    df.loc[mask, col] = np.nan

# Clip physical bounds
df['humidity_day'] = df['humidity_day'].clip(0, 100)
df['humidity_night'] = df['humidity_night'].clip(0, 100)
df['wind'] = df['wind'].clip(0, None)
df['precipitation'] = df['precipitation'].clip(0, None)

print(f'Dataset shape: {df.shape}')
print(f'\nMissing values:\n{df.isnull().sum()}')
df.head()


## 2. Exploratory Data Analysis


In [ ]:
print('=== Descriptive Statistics ===')
display(df.describe())

# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df['weather_label'].value_counts()
axes[0].bar(counts.index, counts.values, color=['#F4D03F', '#85C1E9', '#5DADE2', '#1A5276'])
axes[0].set_title('Weather Condition Class Distribution')
axes[0].set_xlabel('Weather Label')
axes[0].set_ylabel('Count')
for i, (idx, val) in enumerate(counts.items()):
    axes[0].text(i, val + 5, str(val), ha='center', fontweight='bold')

# Correlation matrix (numeric columns only)
numeric_cols = ['precipitation', 'temp_max', 'temp_min', 'humidity_day', 'humidity_night', 'wind']
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1], square=True)
axes[1].set_title('Feature Correlation Matrix')

plt.tight_layout()
plt.show()

# Feature distributions by label
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
colors = {'Sunny': '#F4D03F', 'Cloudy': '#85C1E9', 'Rainy': '#5DADE2', 'Stormy': '#1A5276'}
for ax, col in zip(axes, numeric_cols):
    for lbl, grp in df.groupby('weather_label'):
        grp[col].dropna().hist(ax=ax, alpha=0.6, label=lbl, bins=25, color=colors.get(lbl))
    ax.set_title(f'Distribution of {col}')
    ax.set_xlabel(col)
    ax.legend(fontsize=8)
plt.suptitle('Feature Distributions by Weather Label', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()


## 3. Preprocessing
- NOCB (Next Observation Carried Backward) fill for missing values
- `LabelEncoder` for target
- `StandardScaler` for features


In [ ]:
feature_cols = ['precipitation', 'temp_max', 'temp_min', 'humidity_day', 'humidity_night', 'wind']

# NOCB fill: sort by index, backward fill, then forward fill for leading NaNs
df[feature_cols] = df[feature_cols].bfill().ffill()
print(f'Missing values after NOCB fill: {df[feature_cols].isnull().sum().sum()}')

# Encode target
le = LabelEncoder()
y = le.fit_transform(df['weather_label'])
print(f'Classes: {list(le.classes_)}')
print(f'Encoded labels: {np.unique(y)}')

# Scale features
X_raw = df[feature_cols].values
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

print(f'\nFeature matrix shape: {X.shape}')
print(f'Target vector shape:  {y.shape}')


## 4. Train / Test Split (80 / 20)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}')


## 5. Model Comparison
Train GBC (primary) + RF, LogReg, DT, SVM, KNN and tabulate accuracy.


In [ ]:
classifiers = {
    'Gradient Boosting (GBC)': GradientBoostingClassifier(n_estimators=200, max_depth=4,
                                                           learning_rate=0.1, random_state=42),
    'Random Forest (RF)':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Logistic Regression':      LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree (DT)':       DecisionTreeClassifier(max_depth=8, random_state=42),
    'SVM (RBF)':                SVC(kernel='rbf', C=1.0, random_state=42),
    'KNN (k=5)':                KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
}

results = []
trained_models = {}
for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, clf.predict(X_train))
    test_acc  = accuracy_score(y_test,  clf.predict(X_test))
    results.append({'Model': name, 'Train Accuracy': train_acc, 'Test Accuracy': test_acc})
    trained_models[name] = clf
    print(f'{name:<35}  train={train_acc:.4f}  test={test_acc:.4f}')

results_df = pd.DataFrame(results).sort_values('Test Accuracy', ascending=False)
print('\n=== Accuracy Summary ===')
display(results_df.style.highlight_max(subset=['Test Accuracy'], color='lightgreen'))


## 6. Confusion Matrix (GBC)


In [ ]:
best_model = trained_models['Gradient Boosting (GBC)']
y_pred = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_title('Confusion Matrix — Gradient Boosting Classifier', fontsize=14)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.tight_layout()
plt.show()

print('\n=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=le.classes_))


## 7. Save Artifacts


In [ ]:
artifacts = {
    'weather_gbc.pkl':           best_model,
    'weather_scaler.pkl':        scaler,
    'weather_label_encoder.pkl': le
}

for filename, obj in artifacts.items():
    path = os.path.join(ARTIFACTS_DIR, filename)
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    print(f'Saved: {path}')

print('\nAll weather model artifacts saved successfully.')
